In [1]:
# MNIST dataset으로 CNN 실습 - 함수형 API
import tensorflow as tf
print(tf.__version__)   # 2.20.0
import numpy as np
import matplotlib.pyplot as plt

# 1) 데이터 준비
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
print(x_train[0])
print(x_test.shape)

# 채널(channel) 추가 (흑백인 경우 1)
x_train = x_train.reshape((-1, 28, 28, 1)).astype('float') / 255.0   # 정규화
x_test = x_test.reshape((-1, 28, 28, 1)).astype('float') / 255.0
print(x_train.shape)   # (60000, 28, 28, 1)

2.20.0
[[  0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0]
 [  0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0]
 [  0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0]
 [  0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0]
 [  0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0]
 [  0   0   0   0   0   0   0   0   0   0   0   0   3  18  18  18 126 136
  175  26 166 255 247 127   0   0   0   0]
 [  0   0   0   0   0   0   0   0  30  36  94 154 170 253 253 253 253 253
  225 172 253 242 195  64   0   0   0   0]
 [  0   0   0   0   0   0   0  49 238 253 253 253 253 253 253 253 253 251
   93  82  82  56  39   0   0   0   0   0]
 [  0   0   0   0   0   0   0  18 219 253 253 253 253 253

In [3]:
# 모델 정의 1
"""
inputs = tf.keras.layers.Input(shape=(28, 28, 1))

# Convolution Block1
x = tf.keras.layers.Conv2D(filters=16, kernel_size=(3,3), padding='same', activation='relu')(inputs)
x = tf.keras.layers.MaxPool2D(pool_size=(2,2))(x)
x = tf.keras.layers.Dropout(rate=0.2)(x)

# Convolution Block2
x = tf.keras.layers.Conv2D(filters=32, kernel_size=(3,3), padding='same', activation='relu')(x)
x = tf.keras.layers.MaxPool2D(pool_size=(2,2))(x)
x = tf.keras.layers.Dropout(rate=0.2)(x)

# Fully Connected Layers
x = tf.keras.layers.Flatten()(x)

x = tf.keras.layers.Dense(units=64, activation='relu')(x)
x = tf.keras.layers.Dropout(rate=0.2)(x)
x = tf.keras.layers.Dense(units=32, activation='relu')(x)
x = tf.keras.layers.Dropout(rate=0.2)(x)
outputs = tf.keras.layers.Dense(units=10, activation='softmax')(x)
"""

# 모델 정의 2
inputs = tf.keras.layers.Input(shape=(28, 28, 1))

# Convolution Block1
# use_bias=False : RelU 레이어 분리 -> BatchNormalization 뒤에 Relu를 둠
# BatchNormalization : 과적합 방지 - 학습 안정화, 수렴 가속화
x = tf.keras.layers.Conv2D(filters=16, kernel_size=(3,3), padding='same', use_bias=False)(inputs)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.ReLU()(x)
x = tf.keras.layers.MaxPool2D(pool_size=(2,2))(x)
x = tf.keras.layers.Dropout(rate=0.2)(x)

# Convolution Block2
x = tf.keras.layers.Conv2D(filters=64, kernel_size=(3,3), padding='same', use_bias=False)(x)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.ReLU()(x)
x = tf.keras.layers.MaxPool2D(pool_size=(2,2))(x)
x = tf.keras.layers.Dropout(rate=0.2)(x)

# Fully Connected Layers
x = tf.keras.layers.Flatten()(x)

x = tf.keras.layers.Dense(units=64, activation='relu')(x)
x = tf.keras.layers.Dropout(rate=0.2)(x)
x = tf.keras.layers.Dense(units=32, activation='relu')(x)
x = tf.keras.layers.Dropout(rate=0.2)(x)
outputs = tf.keras.layers.Dense(units=10, activation='softmax')(x)

model = tf.keras.models.Model(inputs=inputs, outputs=outputs, name='mnist_cnn_func')
print(model.summary())


Model: "mnist_cnn_func"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 28, 28, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 28, 28, 16)     │           144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 28, 28, 16)     │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_4 (ReLU)                  │ (None, 28, 28, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 14, 14, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 14, 14, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 14, 14, 64)     │         9,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 14, 14, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_5 (ReLU)                  │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 3136)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 64)             │       200,768 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 10)             │           330 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 212,858 (831.48 KB)

 Trainable params: 212,698 (830.85 KB)

 Non-trainable params: 160 (640.00 B)

None


In [4]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

es = tf.keras.callbacks.EarlyStopping(monitor= 'val_loss', patience=3, restore_best_weights=True)

history = model.fit(
    x_train, y_train, epochs=100, batch_size=128, validation_split=0.1, callbacks=[es], verbose=1
)
# 모델 평가 : 아래 둘의 평가 점수의 차이가 크면 과적합 의심
train_loss, train_acc = model.evaluate(x_train, y_train, verbose=0)
print(f'Train Loss: {train_loss:.4f}, train Accyuracy: {train_acc:.4f}')
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f'Test Loss: {test_loss:.4f}, Test Accyuracy: {test_acc:.4f}')

Epoch 1/100
422/422 ━━━━━━━━━━━━━━━━━━━━ 17s 18ms/step - accuracy: 0.7694 - loss: 0.7069 - val_accuracy: 0.9520 - val_loss: 0.2425
Epoch 2/100
422/422 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.9113 - loss: 0.2907 - val_accuracy: 0.9813 - val_loss: 0.0699
Epoch 3/100
422/422 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.9313 - loss: 0.2294 - val_accuracy: 0.9853 - val_loss: 0.0615
Epoch 4/100
422/422 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9396 - loss: 0.1998 - val_accuracy: 0.9860 - val_loss: 0.0496
Epoch 5/100
422/422 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9464 - loss: 0.1801 - val_accuracy: 0.9865 - val_loss: 0.0457
Epoch 6/100
422/422 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9493 - loss: 0.1676 - val_accuracy: 0.9867 - val_loss: 0.0481
Epoch 7/100
422/422 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9538 - loss: 0.1570 - val_accuracy: 0.9883 - val_loss: 0.0463
Epoch 8/100
422/422 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.9574 - loss: 0.1447 - val_ac